# Iris Model Monitoring & Observability

## Scenario
A classifier is trained on the Iris dataset. Later, incoming flower measurements shift: petal lengths and widths become larger than the model's reference data. This notebook simulates that feature drift and checks statistical drift, rolling windows, species segments, model quality, and monitoring signals.

**Scope:** Iris is a small educational dataset. The later-period rows are intentionally transformed synthetic inputs, not a real production feed. The repository's Docker service serves the Iris model.

## 1. Setup
Install the repository dependencies from the project root with `python -m pip install -r requirements.txt`, then select that environment as this notebook kernel.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

SEED = 42
FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
iris = load_iris(as_frame=True)
X = iris.data.copy()
X.columns = FEATURES
y = iris.target.map(dict(enumerate(iris.target_names))).rename("species")
print(f"Rows: {len(X)} | Features: {FEATURES} | Classes: {sorted(y.unique())}")
display(pd.concat([X, y], axis=1).head())

Rows: 150 | Features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width'] | Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## 2. Train a reference model and record baseline quality
Use a stratified split so each Iris species is represented in both training and reference samples.

In [2]:
X_train, X_reference, y_train, y_reference = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(X_train, y_train)

def quality_metrics(actual, predicted):
    return {
        "accuracy": accuracy_score(actual, predicted),
        "precision_weighted": precision_score(actual, predicted, average="weighted", zero_division=0),
        "recall_weighted": recall_score(actual, predicted, average="weighted", zero_division=0),
        "f1_weighted": f1_score(actual, predicted, average="weighted", zero_division=0),
    }

y_reference_pred = model.predict(X_reference)
print("Reference holdout quality")
display(pd.Series(quality_metrics(y_reference, y_reference_pred), name="score").to_frame())
print(classification_report(y_reference, y_reference_pred, zero_division=0))

Reference holdout quality


,score
accuracy,0.911111
precision_weighted,0.915535
recall_weighted,0.911111
f1_weighted,0.910714


              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.82      0.93      0.88        15
   virginica       0.92      0.80      0.86        15

    accuracy                           0.91        45
   macro avg       0.92      0.91      0.91        45
weighted avg       0.92      0.91      0.91        45



## 3. Simulate an Iris feature shift
Keep the same held-out rows and labels, then increase petal measurements to create a controlled covariate shift. This is deliberately synthetic; labels are retained only so model quality can be compared.

In [3]:
def apply_petal_shift(frame):
    shifted = frame.copy()
    shifted["petal_length"] += 1.0
    shifted["petal_width"] += 0.4
    return shifted

X_production = apply_petal_shift(X_reference)
y_production = y_reference.copy()
y_production_pred = model.predict(X_production)
quality_comparison = pd.DataFrame(
    {
        "reference": quality_metrics(y_reference, y_reference_pred),
        "shifted_period": quality_metrics(y_production, y_production_pred),
    }
).T

display(quality_comparison.round(3))

,accuracy,precision_weighted,recall_weighted,f1_weighted
reference,0.911,0.916,0.911,0.911
shifted_period,0.667,0.685,0.667,0.630


## 4. Statistical drift: PSI and Kolmogorov-Smirnov
PSI compares binned feature distributions. The two-sample KS test compares each numeric feature without binning. Treat thresholds as operational heuristics that should be calibrated for the use case.

In [4]:
def population_stability_index(reference, current, bins=10):
    reference = np.asarray(reference, dtype=float)
    current = np.asarray(current, dtype=float)
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 2:
        raise ValueError("Reference data must contain at least two distinct values")
    edges[0], edges[-1] = -np.inf, np.inf
    ref_share = np.histogram(reference, bins=edges)[0] / len(reference)
    cur_share = np.histogram(current, bins=edges)[0] / len(current)
    eps = 1e-6
    return float(np.sum((cur_share - ref_share) * np.log((cur_share + eps) / (ref_share + eps))))

drift_rows = []
for feature in FEATURES:
    ks = ks_2samp(X_reference[feature], X_production[feature])
    psi = population_stability_index(X_reference[feature], X_production[feature])
    drift_rows.append({"feature": feature, "PSI": psi, "KS statistic": ks.statistic, "KS p-value": ks.pvalue})

drift_summary = pd.DataFrame(drift_rows).set_index("feature")
display(drift_summary.round(4))

,PSI,KS statistic,KS p-value
feature,,,
sepal_length,0.0000,0.0000,1.000
sepal_width,0.0000,0.0000,1.000
petal_length,6.2960,0.3778,0.003
petal_width,5.7573,0.3333,0.013


## 5. Window-based monitoring
Compare consecutive windows against the reference distribution to see when the shift begins. The stream below has stable windows first, followed by shifted windows.

In [5]:
rng = np.random.default_rng(SEED)
window_size = 60
windows = []
for window_id in range(4):
    sampled = X_reference.sample(n=window_size, replace=True, random_state=SEED + window_id).reset_index(drop=True)
    if window_id >= 2:
        sampled = apply_petal_shift(sampled)
    windows.append((window_id + 1, sampled))

window_rows = []
for window_id, window in windows:
    for feature in FEATURES:
        window_rows.append({
            "window": window_id,
            "feature": feature,
            "PSI": population_stability_index(X_reference[feature], window[feature]),
        })
window_summary = pd.DataFrame(window_rows)
display(window_summary.pivot(index="window", columns="feature", values="PSI").round(3))

feature,petal_length,petal_width,sepal_length,sepal_width
window,,,,
1,0.159,0.046,0.150,0.081
2,0.028,0.083,0.077,0.029
3,6.465,6.006,0.085,0.139
4,6.303,7.134,0.179,0.262


## 6. Domain-based monitoring by Iris species
A global average can hide localized changes. Since this controlled exercise has species labels, compare feature drift within each species. In a real unlabeled production stream, use an available segment such as device, geography, or model-score band instead.

In [6]:
segment_rows = []
for species in sorted(y_reference.unique()):
    ref_mask = y_reference.reset_index(drop=True) == species
    cur_mask = y_production.reset_index(drop=True) == species
    for feature in FEATURES:
        segment_rows.append({
            "species": species,
            "feature": feature,
            "PSI": population_stability_index(
                X_reference.reset_index(drop=True).loc[ref_mask, feature],
                X_production.reset_index(drop=True).loc[cur_mask, feature],
            ),
        })
segment_summary = pd.DataFrame(segment_rows)
display(segment_summary.pivot(index="species", columns="feature", values="PSI").round(3))

# Target-distribution check: Pearson chi-square on reference vs shifted-period species counts.
class_counts = pd.DataFrame({
    "reference": y_reference.value_counts(),
    "shifted_period": y_production.value_counts(),
}).fillna(0).astype(int)
chi2, p_value, _, _ = chi2_contingency(class_counts.to_numpy())
print(f"Species-label distribution chi-square p-value: {p_value:.4f}")

feature,petal_length,petal_width,sepal_length,sepal_width
species,,,,
setosa,10.948,13.224,0.0,0.0
versicolor,9.192,6.624,0.0,0.0
virginica,8.985,4.061,0.0,0.0


Species-label distribution chi-square p-value: 1.0000


## 7. Optional Evidently drift report
This creates an HTML data-drift report from the same Iris feature tables. It is an offline report, not a live integration with the container service.

In [8]:
import importlib.util
from pathlib import Path

if importlib.util.find_spec("evidently") is None:
    print("Evidently is not installed in this kernel. Install it with `python -m pip install -r requirements.txt` to create this optional report.")
else:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset

    report = Report(metrics=[DataDriftPreset()])
    report.run(reference_data=X_reference.reset_index(drop=True), current_data=X_production.reset_index(drop=True))
    report_path = Path("notebooks/generated/iris_data_drift_report.html")
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report.save_html(str(report_path))
    print(f"Saved Evidently report to {report_path}")

Saved Evidently report to notebooks\generated\iris_data_drift_report.html


## 8. Service metrics, dashboards, and alerts

The repository's Iris API is served at `http://localhost:8000`; its `/metrics` endpoint exports:

- `model_requests_total{outcome=...}` for request success/error counts
- `model_request_duration_seconds` for request latency histogram buckets
- `model_predictions_total{prediction=...}` for predicted Iris classes
- `model_feature_psi{feature=...}` for recent-input PSI against the Iris reference

After starting the local Compose stack, inspect `http://localhost:9090` and `http://localhost:3000`. The provisioned Grafana dashboard visualizes request rate, error ratio, p95 latency, and feature PSI. The notebook's altered measurements are an educational simulation; send similarly shifted feature rows to `/invocations` to exercise the live API window. Host CPU/memory and distributed traces are not emitted by this service and require a host exporter / OpenTelemetry instrumentation.

## Takeaways
- Statistical drift (PSI and KS), time-window monitoring, and species-segment analysis are demonstrated on Iris features.
- Compare feature drift with prediction quality: drift is a signal to investigate, not proof that a model has failed.
- The Docker API emits real request, latency, prediction, and feature-PSI metrics; notebook labels and simulated windows are not production telemetry.